# S1.P4 GitHub CI 回归修复记录

读取步骤汇总配置生成状态表，不执行测试或读取数据集。记录 PR 前两次 Linux CI 的具体结果以及本地修正后的全门禁结果。

In [1]:
from pathlib import Path
import hashlib, yaml
config_path = Path("modules/m1_data_selection/configs/s1p4_ci_regression_followup.yaml")
config_hash = hashlib.sha256(config_path.read_bytes()).hexdigest()
result = yaml.safe_load(config_path.read_text())
print("config:", config_path, "sha256:", config_hash)
print("seed: not applicable; this notebook summarizes engineering checks")
print("git: source change tracked by", result["change_id"])

config: modules/m1_data_selection/configs/s1p4_ci_regression_followup.yaml sha256: 9fcc00e4f2329c8979acbc9e0634719cc546dc9bebfd6179cba53bfb1ca989cf
seed: not applicable; this notebook summarizes engineering checks
git: source change tracked by CL-20260929-M1-003


In [2]:
from IPython.display import display, Markdown
lines = ["## 本地完整门禁（修复后）", "", "| 项目 | 结果 |", "|---|---|"]
for name, value in result["local_full_gates_after_fixes"].items():
    lines.append(f"| {name} | {value} |")
lines += ["", "## GitHub 首两次运行", "", "| 运行 | 状态与发现 |", "|---|---|"]
for key in ("github_run_1", "github_run_2"):
    lines.append(f"| {key} | {result[key]} |")
lines += ["", "## 修复", "", *[f"- {v}" for v in result["remediation"].values()], "", result["follow_up"]]
display(Markdown("\n".join(lines)))

## 本地完整门禁（修复后）

| 项目 | 结果 |
|---|---|
| status | passed |
| tests | 164 passed |
| core_line_coverage | 96.1% (1813/1887) |
| gate_regression | 47 passed; 0 failed |
| gate6_warnings | 4 documented privacy-cleared notebook warnings |

## GitHub 首两次运行

| 运行 | 状态与发现 |
|---|---|
| github_run_1 | {'commit': 'pre-S1.P3 Ruff pin', 'status': 'failed', 'finding': 'Ruff unpinned; gate 5 Q1 failed; Linux code-quality job had 163 tests pass, 1 native macOS-only test skipped, 96.1% coverage.'} |
| github_run_2 | {'commit': '7e303cd', 'status': 'failed', 'gate_regression': '45 passed; 2 failed', 'findings': ['Pull request regression job checked out GitHub synthetic merge commit, so gate 2 HEAD self-check saw merge and skipped.', 'Latest unpinned MyPy rejected absent platform/ fixture directory; gate 5-B regression expected pass.']} |

## 修复

- requirements.txt pinned to 0.12.0; matching committed lock
- requirements.txt pinned to 1.17.1; matching committed lock
- only passes existing modules/ and platform/ roots to MyPy; reports a hard failure if neither exists
- gate regression job checks out pull request head SHA, not synthetic merge ref
- updated test count from 33 to 47

GitHub Actions will rerun the full workflow on this commit. Check the PR for live status; local tests do not substitute for remote results.